## Problem 6.2 - Bias - Variance Decomposition - Ridge Regression

In [1]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

In [2]:
# define set-up
np.random.seed(42)
sigma = 0.1
degree = 5
n = 20 
n1 = 100
n2 = 500

lamdas = np.power(10,np.linspace(-5,1,100))
poly = PolynomialFeatures(degree = degree, include_bias = True)

theta_star = np.zeros(degree + 1)
theta_star[2] = 1  # coefficient for x^2 term

I = np.eye(degree + 1)

In [3]:
def f(x):
    return x**2

In [4]:
avg_empirical_means = []
avg_empirical_std = []
avg_theoratical_bias = []
avg_theoretical_variance = []
avg_theroretical_risks = []

In [ ]:
for lam in lamdas:
    empirical_risks = []
    empirical_std = []
    theoretical_bias = []
    theoretical_variance = []
    theroretical_risks = []

    for _ in range(n1):
        x_train = np.random.uniform(-1, 1, size = n)
        phi = poly.fit_transform(x_train.reshape(-1,1))
        sigma_hat = (1 / n) * phi.T @ phi
        
        y_train = f(x_train) + np.random.normal(0, sigma, size = n)
        model = Ridge(alpha = n * lam, fit_intercept = False)
        model.fit(phi, y_train)

        theo_bias = lam**2 * theta_star.T @ np.linalg.solve(sigma_hat + lam * I, sigma_hat)**2 @ sigma_hat @ theta_star
        theo_variance = (sigma**2 / n) * np.trace(sigma_hat**2 @ np.linalg.solve(sigma_hat + lam * I, sigma_hat)**2)
        theoretical_bias.append(theo_bias)
        theoretical_variance.append(theo_variance)
        theroretical_risks.append(theo_bias + theo_variance)
        
        empirical_risks_n2 = []
        empirical_std_n2 = []
        for _ in range(n2):
            x_test = np.random.uniform(-1, 1, size = n)
            phi_test = poly.fit_transform(x_test.reshape(-1,1))
            y_test = f(x_test) + np.random.normal(0, sigma, size = n)
            y_pred = model.predict(phi_test)
            empirical_risks_n2.append(np.mean((y_test - y_pred)**2))

        empirical_risks.append(np.mean(empirical_risks_n2))
        empirical_std.append(np.std(empirical_risks_n2))
    
    avg_empirical_means.append(np.mean(empirical_risks))
    avg_empirical_std.append(np.mean(empirical_std))
    avg_theoratical_bias.append(np.mean(theoretical_bias))
    avg_theoretical_variance.append(np.mean(theoretical_variance))
    avg_theroretical_risks.append(np.mean(theroretical_risks))

In [ ]:
# plot the results
plt.figure(figsize=(10,6))
plt.semilogx(lamdas, avg_empirical_means, color = 'hotpink')
plt.semilogx(lamdas, avg_theroretical_risks, linestyle = '--', color = 'pink')
plt.xlabel('Regularization Parameter λ')
plt.ylabel('Estimated Risk')
plt.title('Estimated Risk vs Regularization Parameter')
plt.legend(['Empirical Estimated Risk', 'Theoretical Estimated Risk'])
plt.grid(True)
plt.show()
